# 3D Molecular Geometry Reconstruction

Interactive notebook for visualizing original vs reconstructed molecular geometries using the trained GCN model.

In [ ]:
# Install plotly if needed (for interactive 3D visualization)
# !pip install plotly

In [ ]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Import reconstruction functions
from reconstruction import (
    load_trained_model,
    load_molecule_from_csv,
    reconstruct_molecule,
    align_molecules,
    compute_rmsd
)
from utils.config import load_config

print("Imports successful!")

## 1. Load the Trained Model

In [ ]:
# Configuration paths
CHECKPOINT_PATH = "checkpoints/best_model.pt"
CONFIG_PATH = "config.yaml"
CSV_PATH = "data/Compound_000000001_000025000_props_geom.csv"

# Load model
model, element_to_idx, device = load_trained_model(CHECKPOINT_PATH, CONFIG_PATH)
config = load_config(CONFIG_PATH)
max_bonds = config['model']['max_bonds']

print(f"\nModel loaded successfully!")
print(f"Device: {device}")
print(f"Max bonds: {max_bonds}")

## 2. Helper Functions for Interactive 3D Visualization

In [ ]:
# Element colors (CPK coloring scheme)
ELEMENT_COLORS = {
    'H': '#FFFFFF', 'C': '#909090', 'N': '#3050F8', 'O': '#FF0D0D',
    'F': '#90E050', 'S': '#FFFF30', 'Cl': '#1FF01F', 'Br': '#A62929',
    'P': '#FF8000', 'I': '#940094', 'B': '#FFB5B5', 'Si': '#F0C8A0'
}
DEFAULT_COLOR = '#FF1493'

# Element sizes (van der Waals radii scaled)
ELEMENT_SIZES = {
    'H': 15, 'C': 25, 'N': 25, 'O': 25, 'F': 22, 'S': 30,
    'Cl': 28, 'Br': 30, 'P': 28, 'I': 33, 'B': 25, 'Si': 30
}
DEFAULT_SIZE = 25


def create_3d_molecule_trace(coords, elements, bonds_data, name="Molecule", showlegend=True):
    """
    Create Plotly traces for a 3D molecule visualization.
    
    Returns:
        atoms_trace: Scatter3d trace for atoms
        bonds_traces: List of Scatter3d traces for bonds
    """
    # Get colors and sizes
    colors = [ELEMENT_COLORS.get(e, DEFAULT_COLOR) for e in elements]
    sizes = [ELEMENT_SIZES.get(e, DEFAULT_SIZE) for e in elements]
    
    # Create atom labels
    labels = [f"{elem}{i}" for i, elem in enumerate(elements)]
    
    # Atoms trace
    atoms_trace = go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode='markers+text',
        marker=dict(
            size=sizes,
            color=colors,
            line=dict(color='black', width=1),
            opacity=0.9
        ),
        text=labels,
        textposition='top center',
        textfont=dict(size=10),
        name=name,
        showlegend=showlegend,
        hovertemplate='%{text}<br>x: %{x:.3f}<br>y: %{y:.3f}<br>z: %{z:.3f}<extra></extra>'
    )
    
    # Bonds traces
    bonds_traces = []
    for bond in bonds_data:
        if 'i' in bond and 'j' in bond:
            i, j = bond['i'], bond['j']
            if i < len(coords) and j < len(coords):
                bond_trace = go.Scatter3d(
                    x=[coords[i, 0], coords[j, 0]],
                    y=[coords[i, 1], coords[j, 1]],
                    z=[coords[i, 2], coords[j, 2]],
                    mode='lines',
                    line=dict(color='gray', width=5),
                    showlegend=False,
                    hoverinfo='skip'
                )
                bonds_traces.append(bond_trace)
    
    return atoms_trace, bonds_traces


def visualize_comparison_interactive(original_coords, reconstructed_coords, elements, bonds_data, cid, rmsd_value):
    """
    Create an interactive side-by-side 3D visualization using Plotly.
    """
    # Create figure with 2 subplots
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=(f'Original Geometry ({cid})', f'Reconstructed (RMSD: {rmsd_value:.4f} Å)'),
        horizontal_spacing=0.05
    )
    
    # Original molecule
    atoms1, bonds1 = create_3d_molecule_trace(original_coords, elements, bonds_data, "Original")
    fig.add_trace(atoms1, row=1, col=1)
    for bond in bonds1:
        fig.add_trace(bond, row=1, col=1)
    
    # Reconstructed molecule
    atoms2, bonds2 = create_3d_molecule_trace(reconstructed_coords, elements, bonds_data, "Reconstructed")
    fig.add_trace(atoms2, row=1, col=2)
    for bond in bonds2:
        fig.add_trace(bond, row=1, col=2)
    
    # Calculate axis ranges
    all_coords = np.vstack([original_coords, reconstructed_coords])
    max_range = np.max(np.ptp(all_coords, axis=0)) / 2 * 1.2
    mid = all_coords.mean(axis=0)
    
    # Common scene settings
    scene_settings = dict(
        xaxis=dict(range=[mid[0]-max_range, mid[0]+max_range], title='X (Å)'),
        yaxis=dict(range=[mid[1]-max_range, mid[1]+max_range], title='Y (Å)'),
        zaxis=dict(range=[mid[2]-max_range, mid[2]+max_range], title='Z (Å)'),
        aspectmode='cube'
    )
    
    fig.update_layout(
        title=dict(text=f'Molecular Reconstruction: {cid}', x=0.5),
        scene=scene_settings,
        scene2=scene_settings,
        height=600,
        width=1200,
        showlegend=True
    )
    
    return fig


def visualize_overlay_interactive(original_coords, reconstructed_coords, elements, bonds_data, cid, rmsd_value):
    """
    Create an interactive overlay 3D visualization (both structures in same view).
    """
    fig = go.Figure()
    
    # Original molecule (solid)
    atoms1, bonds1 = create_3d_molecule_trace(original_coords, elements, bonds_data, "Original", showlegend=True)
    fig.add_trace(atoms1)
    for bond in bonds1:
        bond.line.color = 'blue'
        fig.add_trace(bond)
    
    # Reconstructed molecule (semi-transparent)
    atoms2, bonds2 = create_3d_molecule_trace(reconstructed_coords, elements, bonds_data, "Reconstructed", showlegend=True)
    atoms2.marker.opacity = 0.5
    atoms2.marker.line.color = 'red'
    atoms2.marker.line.width = 2
    fig.add_trace(atoms2)
    for bond in bonds2:
        bond.line.color = 'red'
        bond.line.width = 3
        fig.add_trace(bond)
    
    # Calculate axis ranges
    all_coords = np.vstack([original_coords, reconstructed_coords])
    max_range = np.max(np.ptp(all_coords, axis=0)) / 2 * 1.2
    mid = all_coords.mean(axis=0)
    
    fig.update_layout(
        title=dict(text=f'Overlay: {cid} (RMSD: {rmsd_value:.4f} Å)<br><sup>Blue=Original, Red=Reconstructed</sup>', x=0.5),
        scene=dict(
            xaxis=dict(range=[mid[0]-max_range, mid[0]+max_range], title='X (Å)'),
            yaxis=dict(range=[mid[1]-max_range, mid[1]+max_range], title='Y (Å)'),
            zaxis=dict(range=[mid[2]-max_range, mid[2]+max_range], title='Z (Å)'),
            aspectmode='cube'
        ),
        height=700,
        width=900,
        showlegend=True
    )
    
    return fig


print("Visualization functions loaded!")

## 3. Reconstruction Function

In [ ]:
def process_molecule(molecule_idx, csv_path=CSV_PATH):
    """
    Load and reconstruct a molecule by index.
    
    Args:
        molecule_idx: Index of molecule in the CSV (0-based, among valid molecules)
        csv_path: Path to the CSV file
    
    Returns:
        dict with original_coords, reconstructed_coords, elements, bonds, cid, rmsd
    """
    print(f"Loading molecule {molecule_idx}...")
    
    # Load molecule
    coords, elements, bonds, cid = load_molecule_from_csv(csv_path, molecule_idx)
    original_coords = np.array(coords)
    
    print(f"  CID: {cid}")
    print(f"  Atoms: {len(elements)}")
    print(f"  Bonds: {len([b for b in bonds if 'i' in b and 'j' in b])}")
    print(f"  Elements: {set(elements)}")
    
    # Reconstruct
    print("  Reconstructing...")
    reconstructed_coords = reconstruct_molecule(
        model, coords, elements, bonds, element_to_idx, device, max_bonds
    )
    
    # Align
    aligned_coords = align_molecules(reconstructed_coords, original_coords)
    
    # Compute RMSD
    rmsd_value = compute_rmsd(aligned_coords, original_coords)
    print(f"  RMSD: {rmsd_value:.4f} Å")
    
    return {
        'original_coords': original_coords,
        'reconstructed_coords': aligned_coords,
        'elements': elements,
        'bonds': bonds,
        'cid': cid,
        'rmsd': rmsd_value
    }

print("Process function ready!")

## 4. Visualize Example Molecules

Change `MOLECULE_INDEX` to explore different molecules from the dataset.

In [ ]:
# Change this index to visualize different molecules (0, 1, 2, ...)
MOLECULE_INDEX = 0

# Process the molecule
result = process_molecule(MOLECULE_INDEX)

In [ ]:
# Side-by-side comparison
fig_comparison = visualize_comparison_interactive(
    result['original_coords'],
    result['reconstructed_coords'],
    result['elements'],
    result['bonds'],
    result['cid'],
    result['rmsd']
)
fig_comparison.show()

In [ ]:
# Overlay view (both structures in same coordinate system)
fig_overlay = visualize_overlay_interactive(
    result['original_coords'],
    result['reconstructed_coords'],
    result['elements'],
    result['bonds'],
    result['cid'],
    result['rmsd']
)
fig_overlay.show()

## 5. Compare Multiple Molecules

In [ ]:
# Process multiple molecules
molecule_indices = [0, 1, 2]  # Change these to explore different molecules

results = []
for idx in molecule_indices:
    try:
        result = process_molecule(idx)
        results.append(result)
        print()
    except Exception as e:
        print(f"Error processing molecule {idx}: {e}")

# Summary table
print("\n" + "="*50)
print("Summary")
print("="*50)
for r in results:
    print(f"  {r['cid']}: {len(r['elements'])} atoms, RMSD={r['rmsd']:.4f} Å")

if results:
    avg_rmsd = np.mean([r['rmsd'] for r in results])
    print(f"\n  Average RMSD: {avg_rmsd:.4f} Å")

In [ ]:
# Visualize each molecule from results
for i, result in enumerate(results):
    print(f"\nMolecule {i+1}: {result['cid']}")
    fig = visualize_comparison_interactive(
        result['original_coords'],
        result['reconstructed_coords'],
        result['elements'],
        result['bonds'],
        result['cid'],
        result['rmsd']
    )
    fig.show()

## 6. Analyze Per-Atom Deviations

In [1]:
def analyze_deviations(result):
    """
    Analyze per-atom deviations between original and reconstructed structures.
    """
    original = result['original_coords']
    reconstructed = result['reconstructed_coords']
    elements = result['elements']
    
    # Calculate per-atom distances
    distances = np.linalg.norm(original - reconstructed, axis=1)
    
    # Create DataFrame
    df = pd.DataFrame({
        'Atom': [f"{elem}{i}" for i, elem in enumerate(elements)],
        'Element': elements,
        'Deviation (Å)': distances
    })
    
    print(f"\nPer-Atom Deviations for {result['cid']}:")
    print(df.to_string(index=False))
    
    print(f"\nStatistics:")
    print(f"  Min deviation:  {distances.min():.4f} Å")
    print(f"  Max deviation:  {distances.max():.4f} Å")
    print(f"  Mean deviation: {distances.mean():.4f} Å")
    print(f"  Std deviation:  {distances.std():.4f} Å")
    
    # Plot histogram
    fig = go.Figure(data=[go.Histogram(x=distances, nbinsx=20)])
    fig.update_layout(
        title=f'Distribution of Atomic Deviations ({result["cid"]})',
        xaxis_title='Deviation (Å)',
        yaxis_title='Count',
        height=400,
        width=600
    )
    fig.show()
    
    return df

# Analyze the first result
if results:
    deviation_df = analyze_deviations(results[0])

NameError: name 'results' is not defined

## 7. Visualize with Deviation-Based Coloring

In [2]:
def visualize_with_deviation_coloring(result):
    """
    Visualize the reconstructed structure with atoms colored by deviation.
    """
    original = result['original_coords']
    reconstructed = result['reconstructed_coords']
    elements = result['elements']
    bonds = result['bonds']
    
    # Calculate deviations
    deviations = np.linalg.norm(original - reconstructed, axis=1)
    
    # Create labels
    labels = [f"{elem}{i}<br>Dev: {dev:.3f} Å" for i, (elem, dev) in enumerate(zip(elements, deviations))]
    
    # Create figure
    fig = go.Figure()
    
    # Atoms with deviation coloring
    fig.add_trace(go.Scatter3d(
        x=reconstructed[:, 0],
        y=reconstructed[:, 1],
        z=reconstructed[:, 2],
        mode='markers+text',
        marker=dict(
            size=20,
            color=deviations,
            colorscale='RdYlGn_r',  # Red=high deviation, Green=low
            colorbar=dict(title='Deviation (Å)'),
            line=dict(color='black', width=1)
        ),
        text=[f"{elem}{i}" for i, elem in enumerate(elements)],
        textposition='top center',
        hovertext=labels,
        hoverinfo='text',
        name='Atoms'
    ))
    
    # Bonds
    for bond in bonds:
        if 'i' in bond and 'j' in bond:
            i, j = bond['i'], bond['j']
            if i < len(reconstructed) and j < len(reconstructed):
                fig.add_trace(go.Scatter3d(
                    x=[reconstructed[i, 0], reconstructed[j, 0]],
                    y=[reconstructed[i, 1], reconstructed[j, 1]],
                    z=[reconstructed[i, 2], reconstructed[j, 2]],
                    mode='lines',
                    line=dict(color='gray', width=4),
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Layout
    all_coords = reconstructed
    max_range = np.max(np.ptp(all_coords, axis=0)) / 2 * 1.2
    mid = all_coords.mean(axis=0)
    
    fig.update_layout(
        title=f'Deviation Map: {result["cid"]} (RMSD: {result["rmsd"]:.4f} Å)<br><sup>Green=Low deviation, Red=High deviation</sup>',
        scene=dict(
            xaxis=dict(range=[mid[0]-max_range, mid[0]+max_range], title='X (Å)'),
            yaxis=dict(range=[mid[1]-max_range, mid[1]+max_range], title='Y (Å)'),
            zaxis=dict(range=[mid[2]-max_range, mid[2]+max_range], title='Z (Å)'),
            aspectmode='cube'
        ),
        height=700,
        width=900
    )
    
    return fig

# Visualize with deviation coloring
if results:
    fig_deviation = visualize_with_deviation_coloring(results[0])
    fig_deviation.show()

NameError: name 'results' is not defined

## 8. Save Visualizations

In [ ]:
# Save as interactive HTML files
import os
os.makedirs('reconstruction_results', exist_ok=True)

if results:
    for result in results:
        # Comparison view
        fig = visualize_comparison_interactive(
            result['original_coords'],
            result['reconstructed_coords'],
            result['elements'],
            result['bonds'],
            result['cid'],
            result['rmsd']
        )
        fig.write_html(f"reconstruction_results/{result['cid']}_comparison.html")
        print(f"Saved: reconstruction_results/{result['cid']}_comparison.html")
        
        # Deviation map
        fig_dev = visualize_with_deviation_coloring(result)
        fig_dev.write_html(f"reconstruction_results/{result['cid']}_deviation_map.html")
        print(f"Saved: reconstruction_results/{result['cid']}_deviation_map.html")

## 9. Quick Molecule Viewer

Run the cell below and change the molecule index to quickly explore different molecules.

In [ ]:
# Quick viewer - change index and re-run this cell
QUICK_INDEX = 5  # <-- Change this!

quick_result = process_molecule(QUICK_INDEX)
visualize_comparison_interactive(
    quick_result['original_coords'],
    quick_result['reconstructed_coords'],
    quick_result['elements'],
    quick_result['bonds'],
    quick_result['cid'],
    quick_result['rmsd']
).show()